In [3]:
#!/usr/bin/env python
# coding: utf-8

import os
import sys
import time
import sqlite3
import pandas as pd
import numpy as np

# === 初期設定 ===
start_time = time.time()

# jupyter/py 両対応
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR

# パス定義
if os.name == 'nt':
    user_base = os.path.join(os.environ["USERPROFILE"], "myenv310", PROJECT_DIR)
else:
    user_base = os.path.join(os.path.expanduser("~"), "myenv310", PROJECT_DIR)

db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# 必須列
BASE_NEEDED = ["実行日", "台番号"]
SRC_COLS = ["累計通常ゲーム数時短抜き", "累計出玉pt", "出玉数"]
TARGET_COL = "出玉数時短抜き最終回転率"

with sqlite3.connect(db_path) as conn:
    pragma = pd.read_sql_query("PRAGMA table_info(result_table);", conn)
    table_cols = set(pragma["name"].tolist())
    missing = [c for c in (BASE_NEEDED + SRC_COLS + [TARGET_COL]) if c not in table_cols]
    if missing:
        raise RuntimeError(f"result_table に必要列がありません: {missing}")

    # 日付判定用の最小取得
    df_basic = pd.read_sql_query(
        '''
        SELECT ROWID AS rowid, [実行日], [台番号]
          FROM result_table
        ''',
        conn
    )

# 型整備
df_basic["実行日"] = pd.to_datetime(df_basic["実行日"], errors="coerce")
df_basic = df_basic.dropna(subset=["実行日"])

# 最新日付
max_date = df_basic["実行日"].dt.date.max()

# 各台の最新行rowid
latest_rowid_by_tai = (
    df_basic[df_basic["実行日"].dt.date == max_date]
      .sort_values(["台番号", "実行日", "rowid"], ascending=[True, False, False])
      .groupby("台番号", as_index=False)
      .agg(latest_rowid=("rowid", "max"))
)

# 当日分の必要列取得
select_cols = ["ROWID AS rowid"] + BASE_NEEDED + SRC_COLS + [TARGET_COL]
select_cols_sql = ", ".join([c if c.startswith("ROWID") else f"[{c}]" for c in select_cols])

with sqlite3.connect(db_path) as conn:
    df_today = pd.read_sql_query(
        f'''
        SELECT {select_cols_sql}
          FROM result_table
         WHERE date([実行日]) = ?
        ''',
        conn,
        params=(str(max_date),)
    )

# 最新行のみ
df_latest = df_today[df_today["rowid"].isin(latest_rowid_by_tai["latest_rowid"])].copy()

# 数値化
for c in SRC_COLS:
    df_latest[c] = pd.to_numeric(df_latest[c], errors="coerce")

# === ここが指定の計算式 ===
# 出玉数最終回転率 = 累計通常ゲーム数 ÷ (累計出玉pt – 出玉数) × 250
denom = df_latest["累計出玉pt"] - df_latest["出玉数"]
num = df_latest["累計通常ゲーム数時短抜き"]
rate = np.where((denom > 0) & (~denom.isna()) & (~num.isna()),
                (num / denom) * 250,
                np.nan)

# 丸め不要ならそのまま。桁指定したい場合は .round(2) などを足す
df_latest[TARGET_COL] = pd.to_numeric(rate, errors="coerce")

# DB更新
updated = 0
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    for _, r in df_latest.iterrows():
        val = r[TARGET_COL]
        cur.execute(
            f'''
            UPDATE result_table
               SET [{TARGET_COL}] = ?
             WHERE ROWID = ?
               AND date([実行日]) = ?
            ''',
            (None if pd.isna(val) else float(val), int(r["rowid"]), str(max_date))
        )
        updated += cur.rowcount
    conn.commit()

print(f"✅ {TARGET_COL} 更新完了: {updated} 行（当日各台の最新行のみ）")
print(f"[INFO] 所要時間: {time.time() - start_time:.2f} 秒")


[INFO] 使用DB: C:\Users\stray\myenv310\iwakuni-tekisasu-p\db\output.db
✅ 出玉数時短抜き最終回転率 更新完了: 10 行（当日各台の最新行のみ）
[INFO] 所要時間: 0.02 秒
